# Building `base_KPIs_labels_v3.csv`

This notebook takes the raw CDP `metrics.parquet` file (produced upstream by Chase Hikida's pipeline) and derives a compact, per-respondent-year KPI table: label counts/means for each BERT classifier (environmental claims, climate commitment, specificity, net-zero, sentiment, TCFD, transition, renewable), plus text-fill statistics (`nchar_sum`, `prop_filled`).

The output, `base_KPIs_labels_v3.csv`, is the file merged with Compustat/Trucost financial controls in the main CDP-Compustat-Trucost-BERT merge pipeline.

**Sections:**
1. Load raw metrics data
2. Clean core identifier columns (`account_no`, `year`, `respondent`)
3. Compute `maxcol` (max filled base-survey columns per year, used as a denominator)
4. Helper functions for chunked label counting and text-fill statistics
5. Build and export `base_KPIs_labels_v3.csv`
6. Examples (exploratory queries against the raw/derived data)


## 1. Load raw metrics data

In [1]:
import time
import pandas as pd
import numpy as np
import pyarrow.parquet as pq


In [2]:
start_time = time.time()
print(f"Start: {time.strftime('%Y-%m-%d %H:%M:%S')}")

# metrics.parquet is produced upstream by Chase Hikida's pipeline (see /mse)
df = pq.read_table(source="Chase Hikida/metrics.parquet").to_pandas()  # ~6 minutes

elapsed = time.strftime('%H:%M:%S', time.gmtime(time.time() - start_time))
print(f"Loaded {len(df):,} rows, {len(df.columns):,} columns in {elapsed}")


Start: 2026-08-10 19:59:08
Loaded 41,004 rows, 29,232 columns in 00:08:22


## 2. Clean core identifier columns
Rename the raw CDP column names to short, usable names, then coerce types (the raw parquet stores `year`/`account_no` as list-like strings, e.g. `"['2019']"`).

In [3]:
df.rename(columns={
    'column=Account number|sheet=*|value=response': 'account_no',
    'column=year|sheet=*|value=meta': 'year',
    'column=label|sheet=*|value=meta': 'respondent',
    'column=Organization|sheet=Summary Data|value=response': 'name'
}, inplace=True)


In [4]:
# numeric identifiers: strip [ ] and ' then coerce to numeric
for colname in ['year', 'account_no']:
    df.loc[:, colname] = (
        df[colname]
        .astype(str)
        .str.replace(r"[\[\]']", '', regex=True)
        .pipe(pd.to_numeric, errors='coerce')
    )

# string identifiers: strip [ ] and ' only
for colname in ['respondent']:
    df.loc[:, colname] = (
        df[colname]
        .astype(str)
        .str.replace(r"[\[\]']", '', regex=True)
    )

df[['account_no', 'year', 'respondent']].head()


,account_no,year,respondent
0,200.0,2010,investor
1,1800.0,2010,investor
2,5300.0,2010,investor
3,29900.0,2010,investor
4,28600.0,2010,investor


## 3. Compute `maxcol` per year
`maxcol` is the maximum number of base-survey columns filled by any respondent in a given year. It's used as the denominator when converting label counts into proportions, since the CDP questionnaire length varies by year.

In [5]:
num_filled_cols = [col for col in df.columns if "number_responses" in col]
calc = df[['year'] + num_filled_cols]
calc.loc[:, 'year'] = (
    calc['year']
    .astype(str)
    .str.replace(r"[\[\]']", '', regex=True)
    .pipe(pd.to_numeric, errors='coerce')
)


In [6]:
# max row-sum of filled base-survey columns, per year
year_filled = pd.DataFrame({'year': range(2010, 2021), 'maxcol': range(1, 12)})

for year in year_filled['year']:
    subset = calc[calc['year'] == year]
    row_sums = (subset.drop(columns='year') >= 1).sum(axis=1)
    year_filled.loc[year_filled['year'] == year, 'maxcol'] = (
        row_sums.max() if not row_sums.empty else 0
    )

year_filled


,year,maxcol
0,2010,191
1,2011,241
2,2012,249
3,2013,271
4,2014,293
5,2015,300
6,2016,333
7,2017,336
8,2018,379
9,2019,394


In [7]:
# map maxcol back onto df by year
year_to_maxcol = dict(zip(year_filled['year'], year_filled['maxcol']))
df.loc[:, 'maxcol'] = df['year'].map(year_to_maxcol)


## 4. Helper functions
Both raw metrics and text responses are wide (thousands of columns), so these functions process the DataFrame in row-chunks to keep memory manageable.

In [8]:
def process_chunks_by_pattern(data, cols, pattern, label, n=4000):
    """
    Processes chunks of a DataFrame, calculating:
    1. Row sums of label matches
    2. Row means (sums divided by maxcol values)

    Args:
        data (pd.DataFrame): Input DataFrame.
        cols (list): Columns to process (pre-filtered by pattern).
        pattern (str): [Optional] Pattern used to filter cols.
        label (str): Text label to search for.
        n (int): Rows per chunk.

    Returns:
        tuple: (row_sums, row_means) as concatenated Series.
    """
    row_sums, row_means = [], []
    chunks = [data[i:i + n] for i in range(0, len(data), n)]

    for i, chunk in enumerate(chunks):
        chunk_sum = (
            chunk[cols]
            .astype(str)
            .replace(r"[\[\]']", '', regex=True)
            .apply(lambda col: col.str.contains(label, case=False, na=False))
            .sum(axis=1)
        )
        divisor = chunk['maxcol'].replace(0, 1)  # avoid division by zero
        chunk_mean = chunk_sum / divisor

        print(f"chunk {i} processed")
        row_sums.append(chunk_sum)
        row_means.append(chunk_mean)

    return pd.concat(row_sums, ignore_index=True), pd.concat(row_means, ignore_index=True)


In [9]:
def calc_nchar_sum_prop_filled(data, cols, n=4000):
    """
    Processes chunks of a DataFrame, calculating:
    1. Character count per cell, excluding placeholder text
       ("Question not applicable", "Hidden Answer", "None") and whitespace
    2. Row-wise total character count (nchar_sum)
    3. Proportion of maxcol columns with a non-empty response (prop_filled)

    Args:
        data (pd.DataFrame): Input DataFrame.
        cols (list): Columns to process (pre-filtered by pattern).
        n (int): Rows per chunk.

    Returns:
        tuple: (nchar_sum, prop_filled) as concatenated Series.
    """
    nchar_sum, prop_filled = [], []
    chunks = [data[i:i + n] for i in range(0, len(data), n)]

    for i, chunk in enumerate(chunks):
        cleaned_chunk = (
            chunk[cols].drop('maxcol', axis=1)
            .astype(str)
            .replace(r"[\[\]']", '', regex=True)
            .replace(r"Question not applicable", '', regex=True)
            .replace(r"Hidden Answer", '', regex=True)
            .replace(r"None", '', regex=True)
            .replace(r"\n", "", regex=True)
            .replace(r'\s+', '', regex=True)
        )

        lengths = cleaned_chunk.map(lambda x: len(x))
        sum_lengths = lengths.sum(axis=1)
        nonzero_responses = (lengths >= 1).sum(axis=1)

        divisor = chunk['maxcol'].replace(0, 1)
        filled_proportion = nonzero_responses / divisor

        print(f"chunk {i} processed")
        nchar_sum.append(sum_lengths)
        prop_filled.append(filled_proportion)

    return pd.concat(nchar_sum, ignore_index=True), pd.concat(prop_filled, ignore_index=True)


## 5. Build and export `base_KPIs_labels_v3.csv`
For each BERT classifier, count label matches per row (and convert to a proportion of `maxcol`), then assemble everything alongside the text-fill statistics into `base2`.

In [10]:
start_time = time.time()
print(f"Start: {time.strftime('%Y-%m-%d %H:%M:%S')}")


base2_results = {}

# new since last run (5/23/25)
df_smaller = df[[col for col in df.columns if "column="  in col
  and "sheet=Summary" not in col 
  and "number_characters" not in col
  and "number_responses" not in col
 ] + ['maxcol']]

df_core_survey = df[[col for col in df.columns if "column="  in col
  and "sheet=Summary" not in col 
  and "model=" not in col
  and "number_characters" not in col
  and "number_responses" not in col
 ] + ['maxcol']]

# first, add nchar_sum and prop_filled columns using function I built above
base2_results['nchar_sum'], base2_results['prop_filled'] = calc_nchar_sum_prop_filled(df_core_survey, df_core_survey.columns)


patterns = [#'label=spec|value=score_average', #'number_characters', 'number_responses',
            'model=environmental_claims|value=label', 'model=climate_commitment|value=label',
            'model=climate_specificity|value=label', 'model=netzero_reduction|value=label', 
            'model=climate_sentiment|value=label', 'model=tcfd|value=label', 
            'model=transition|value=label', 'model=renewable|value=label'
]            

for pattern in patterns:
    print(pattern)
    # Find columns matching the pattern, ignore the |, so set regex = False! or use like=pattern
    
    filtered_cols = df_smaller.filter(like=pattern, axis=1).columns
    print(len(filtered_cols))    # CHECKS: there should be 884/886 columns for each 
    
#     if pattern in ['number_characters', 'number_responses']: # 'label=spec|value=score_average',
        
#         if pattern == 'number_responses':
#             base2_results['num_filled'] = df[filtered_cols].map(lambda x: isinstance(x, (int, float)) and x >= 1).sum(axis=1) 
#             base2_results['prop_filled'] = df[filtered_cols].map(lambda x: isinstance(x, (int, float)) and x >= 1).sum(axis=1)  / df['maxcol']
        
#         if pattern in ['number_characters']:
#             base2_results['nchar_sum'] = df[filtered_cols].apply(pd.to_numeric, errors='coerce').sum(axis=1)
#             # I do wonder how this handles the cells that have lists of numbers
        
#         if pattern in ['label=spec|value=score_average']:
#             base2_results['average_specificity_sum'] = df[filtered_cols].apply(pd.to_numeric, errors='coerce').sum(axis=1)
#             base2_results['average_specificity_mean'] = df[filtered_cols].apply(pd.to_numeric, errors='coerce').sum(axis=1) / df['maxcol']

    if pattern in ['model=environmental_claims|value=label', 'model=climate_commitment|value=label']:
        for label in ['yes', 'no']:
            print(label)
            base2_results[f'{pattern}_{label}_sum'], base2_results[f'{pattern}_{label}_mean'] = process_chunks_by_pattern(df,filtered_cols, pattern, label, n=4000)

    if pattern == 'model=climate_specificity|value=label':
        for label in ['spec']:
            print(label)
            base2_results[f'{pattern}_{label}_sum'], base2_results['percent_specificity'] = process_chunks_by_pattern(df,filtered_cols, pattern, label, n=4000)
            # if label == 'spec': / len(filtered_cols) for percent_specificity?
        for label in ['non']:
            print(label)
            base2_results[f'{pattern}_{label}_sum'], base2_results['percent_non_specificity'] = process_chunks_by_pattern(df,filtered_cols, pattern, label, n=4000)
            
    if pattern == 'model=netzero_reduction|value=label':
        for label in ['reduction', 'net-zero']:
            print(label)
            base2_results[f'{pattern}_{label}_sum'],base2_results[f'{pattern}_{label}_mean'] = process_chunks_by_pattern(df,filtered_cols, pattern, label, n=4000)

    if pattern == 'model=climate_sentiment|value=label':
        for label in ['opportunity', 'risk', 'neutral']:
            print(label)
            base2_results[f'{pattern}_{label}_sum'], base2_results[f'{pattern}_{label}_mean'] = process_chunks_by_pattern(df,filtered_cols, pattern, label, n=4000)

    if pattern == 'model=tcfd|value=label':
        for label in ['strategy', 'risk', 'metrics', 'governance']:
            print(label)
            base2_results[f'{pattern}_{label}_sum'],base2_results[f'{pattern}_{label}_mean'] = process_chunks_by_pattern(df,filtered_cols, pattern, label, n=4000)

    if pattern == 'model=transition|value=label':
        for label in ['LABEL_0', 'LABEL_1', 'LABEL_2']:
            print(label)
            base2_results[f'{pattern}_{label}_sum'],base2_results[f'{pattern}_{label}_mean'] = process_chunks_by_pattern(df,filtered_cols, pattern, label, n=4000)

    if pattern == 'model=renewable|value=label':
        for label in ['LABEL_0', 'LABEL_1']:
            print(label)
            base2_results[f'{pattern}_{label}_sum'],base2_results[f'{pattern}_{label}_mean'] = process_chunks_by_pattern(df,filtered_cols, pattern, label, n=4000)
            
base2 = df[['account_no', 'year', 'respondent', 'maxcol']]
base2.loc[:,'year'] = (
    base2['year']
    .astype(str)
    .str.replace(r"[\[\]']", '', regex=True)  # Remove [ ] and '
    .pipe(pd.to_numeric, errors='coerce')     # Convert to numeric
)
base2.loc[:,'account_no'] = (
    base2['account_no']
    .astype(str)
    .str.replace(r"[\[\]']", '', regex=True)  # Remove [ ] and '
    .pipe(pd.to_numeric, errors='coerce')     # Convert to numeric
)
base2.loc[:,'respondent'] = (
    base2['respondent']
    .astype(str)
    .str.replace(r"[\[\]']", '', regex=True)  # Remove [ ] and '
)

# Update base2 DataFrame with the new columns
for col_name, values in base2_results.items():
    base2.loc[:,col_name] = values

end_time = time.time()
print(f"End: {time.strftime('%Y-%m-%d %H:%M:%S')}")

elapsed_seconds = end_time - start_time
elapsed_hms = time.strftime('%H:%M:%S', time.gmtime(elapsed_seconds))
print(f'Elapsed: {elapsed_hms}')

Start: 2026-08-10 20:07:38
chunk 0 processed
chunk 1 processed
chunk 2 processed
chunk 3 processed
chunk 4 processed
chunk 5 processed
chunk 6 processed
chunk 7 processed
chunk 8 processed
chunk 9 processed
chunk 10 processed
model=environmental_claims|value=label
864
yes
chunk 0 processed
chunk 1 processed
chunk 2 processed
chunk 3 processed
chunk 4 processed
chunk 5 processed
chunk 6 processed
chunk 7 processed
chunk 8 processed
chunk 9 processed
chunk 10 processed
no
chunk 0 processed
chunk 1 processed
chunk 2 processed
chunk 3 processed
chunk 4 processed
chunk 5 processed
chunk 6 processed
chunk 7 processed
chunk 8 processed
chunk 9 processed
chunk 10 processed
model=climate_commitment|value=label
864
yes
chunk 0 processed
chunk 1 processed
chunk 2 processed
chunk 3 processed
chunk 4 processed
chunk 5 processed
chunk 6 processed
chunk 7 processed
chunk 8 processed
chunk 9 processed
chunk 10 processed
no
chunk 0 processed
chunk 1 processed
chunk 2 processed
chunk 3 processed
chunk 4

/var/folders/km/qbfhgn7s6y78wdq1sqsfj4fc0000gn/T/ipykernel_46168/2199459185.py:113: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  base2.loc[:,col_name] = values
/var/folders/km/qbfhgn7s6y78wdq1sqsfj4fc0000gn/T/ipykernel_46168/2199459185.py:113: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  base2.loc[:,col_name] = values
/var/folders/km/qbfhgn7s6y78wdq1sqsfj4fc0000gn/T/ipykernel_46168/2199459185.py:113: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc

In [11]:
base2.head()

,account_no,year,respondent,maxcol,nchar_sum,prop_filled,model=environmental_claims|value=label_yes_sum,model=environmental_claims|value=label_yes_mean,model=environmental_claims|value=label_no_sum,model=environmental_claims|value=label_no_mean,...,model=transition|value=label_LABEL_0_sum,model=transition|value=label_LABEL_0_mean,model=transition|value=label_LABEL_1_sum,model=transition|value=label_LABEL_1_mean,model=transition|value=label_LABEL_2_sum,model=transition|value=label_LABEL_2_mean,model=renewable|value=label_LABEL_0_sum,model=renewable|value=label_LABEL_0_mean,model=renewable|value=label_LABEL_1_sum,model=renewable|value=label_LABEL_1_mean
0,200.0,2010,investor,191,4654,0.549738,2,0.010471,862,4.513089,...,22,0.115183,88,0.460733,2,0.010471,105,0.549738,5,0.026178
1,1800.0,2010,investor,191,13446,0.612565,4,0.020942,861,4.507853,...,31,0.162304,93,0.486911,4,0.020942,115,0.602094,16,0.083770
2,5300.0,2010,investor,191,62986,0.816754,1,0.005236,864,4.523560,...,50,0.261780,108,0.565445,11,0.057592,149,0.780105,20,0.104712
3,29900.0,2010,investor,191,51885,0.774869,7,0.036649,861,4.507853,...,50,0.261780,104,0.544503,9,0.047120,148,0.774869,7,0.036649
4,28600.0,2010,investor,191,6207,0.534031,2,0.010471,862,4.513089,...,28,0.146597,76,0.397906,2,0.010471,98,0.513089,9,0.047120


In [12]:
# tidy column names: drop 'model=' prefix, replace '|value=' with '_'
base2.columns = base2.columns.str.replace('model=', '', regex=False).str.replace('|value=', '_', regex=False)

base2.to_csv('preprocessed_data/base_KPIs_labels_v3.csv', index=False)

## 6. Examples
A few common queries against the raw and derived data, kept here for reference.

**Example: load the saved KPI table**

In [13]:
base_KPIs = pd.read_csv('preprocessed_data/base_KPIs_labels_v3.csv')
base_KPIs.head()


,account_no,year,respondent,maxcol,nchar_sum,prop_filled,environmental_claims_label_yes_sum,environmental_claims_label_yes_mean,environmental_claims_label_no_sum,environmental_claims_label_no_mean,...,transition_label_LABEL_0_sum,transition_label_LABEL_0_mean,transition_label_LABEL_1_sum,transition_label_LABEL_1_mean,transition_label_LABEL_2_sum,transition_label_LABEL_2_mean,renewable_label_LABEL_0_sum,renewable_label_LABEL_0_mean,renewable_label_LABEL_1_sum,renewable_label_LABEL_1_mean
0,200.0,2010,investor,191,4654,0.549738,2,0.010471,862,4.513089,...,22,0.115183,88,0.460733,2,0.010471,105,0.549738,5,0.026178
1,1800.0,2010,investor,191,13446,0.612565,4,0.020942,861,4.507853,...,31,0.162304,93,0.486911,4,0.020942,115,0.602094,16,0.083770
2,5300.0,2010,investor,191,62986,0.816754,1,0.005236,864,4.523560,...,50,0.261780,108,0.565445,11,0.057592,149,0.780105,20,0.104712
3,29900.0,2010,investor,191,51885,0.774869,7,0.036649,861,4.507853,...,50,0.261780,104,0.544503,9,0.047120,148,0.774869,7,0.036649
4,28600.0,2010,investor,191,6207,0.534031,2,0.010471,862,4.513089,...,28,0.146597,76,0.397906,2,0.010471,98,0.513089,9,0.047120


**Example: find all Scope 3 emissions-related columns in the raw data**

In [14]:
scope3_cols = [col for col in df.columns if "Scope 3" in col]
scope3_cols[:5]


['column=20.1C3. Scope 3 (Q15.1)|sheet=20.1A|value=response',
 'column=C6.5_C2_Account for your organization’s gross global Scope 3 emissions, disclosing and explaining any exclusions. - Metric tonnes CO2e|sheet=C6.5|value=response',
 'column=C6.5_C3_Account for your organization’s gross global Scope 3 emissions, disclosing and explaining any exclusions. - Emissions calculation methodology|sheet=C6.5|value=response',
 'column=C4.1a_C4_Provide details of your absolute emissions target(s) and progress made against those targets. - Scope(s) (or Scope 3 category)|sheet=C4.1a|value=response',
 'column=C4.1a_C7_Provide details of your absolute emissions target(s) and progress made against those targets. - Covered emissions in base year as % of total base year emissions in selected Scope(s) (or Scope 3 category)|sheet=C4.1a|value=response']

**Example: filter the derived KPI table to a subset of respondents**

In [15]:
account_nos = ['1104', '4657', '5052', '12117', '12942', '16012',
               '16558', '19051', '36707', '52633', '60580', '832087']

mask = base_KPIs['account_no'].astype(str).isin(account_nos)
base_KPIs.loc[mask].head(10)


,account_no,year,respondent,maxcol,nchar_sum,prop_filled,environmental_claims_label_yes_sum,environmental_claims_label_yes_mean,environmental_claims_label_no_sum,environmental_claims_label_no_mean,...,transition_label_LABEL_0_sum,transition_label_LABEL_0_mean,transition_label_LABEL_1_sum,transition_label_LABEL_1_mean,transition_label_LABEL_2_sum,transition_label_LABEL_2_mean,renewable_label_LABEL_0_sum,renewable_label_LABEL_0_mean,renewable_label_LABEL_1_sum,renewable_label_LABEL_1_mean
